In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
#del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

'1'

In [3]:
import json
from urllib.parse import urlparse, unquote
import mwparserfromhell

from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import (
    DatabaseUpdater, 
    _extract_title_links, 
    _fetch_mediawiki_file_metadata_chunk, 
    _normalize_title, 
    _normalize_date_string,
    _get_links,
    _is_category_link,
    _sniff_suffix,
    _form_page_info_from_title,
    )
from birddog.tracker import PageChangeLog
from birddog.utility import fetch_url, transliterate
from birddog.wiki import (
    WIKI_NAMESPACE,
    API_URL,
    expand_link_target,
    canonicalize_title,
    classify_page,
    page_name,
    )

2026-01-14 14:44:00,501 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-01-14 14:44:00,603 [INFO] Translation is enabled. Using GCP translator
2026-01-14 14:44:00,605 [INFO] Using Google Cloud translation API
2026-01-14 14:44:00,605 [INFO] GoogleCloudTranslator using REST API
2026-01-14 14:44:00,878 [INFO] Using local nocodb api: http://localhost:8080/api/v2


In [4]:
def clear_db():
    ids=updater._db.get_all_ids("Documents")
    updater._db.delete("Documents", ids)
    ids=updater._db.get_all_ids("Pages")
    updater._db.delete("Pages", ids)

In [5]:
import random
def deterministic_shuffle(items, seed=42):
    rng = random.Random(seed)   # independent RNG instance
    items = list(items)         # avoid mutating caller’s list
    rng.shuffle(items)
    return items

In [6]:
updater = DatabaseUpdater(Runtime())
with open("var/title_sample.json") as file:
    title_list = json.loads(file.read())
title_list = deterministic_shuffle(sorted(list(set(title_list))))

2026-01-14 14:44:01,165 [INFO] PageUpdateManager.init(): detect_environment==local
2026-01-14 14:44:01,416 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-01-14 14:44:01,416 [INFO] Runtime: truncating log history before 2025-11-15 21:44:01.416880+00:00


In [52]:
clear_db()

In [13]:
updater.clear_alerts()

In [ ]:
updater.update_records(title_list[:50])

In [ ]:
updater.start_translation()

In [9]:
from birddog.store import get_key_value_store

In [10]:
store = get_key_value_store()

In [11]:
int(store.get("DB Loader", "cursor"))

KeyError: 'DB Loader:cursor not found'

In [ ]:
store.remove_all("DB Loader")

In [ ]:
updater.update_records(["foo", title_list[0] + "foo"] + title_list[:3])

In [ ]:
chunk = 100
for i in range(0, len(title_list), chunk):
    print(f"*********** Batch {i} ***********")
    updater.update_records(title_list[i:i+chunk])

In [53]:
from birddog.database_updater import _fetch_mediawiki_file_metadata_chunk

In [54]:
from time import sleep

import json
from birddog.runtime import Runtime
from birddog.database import Database
from birddog.database_updater import DatabaseUpdater
from birddog.tracker import PageChangeLog
from birddog.store import get_key_value_store

updater = DatabaseUpdater(runtime=Runtime())
store = get_key_value_store()
change_log = PageChangeLog()
changes = change_log.get()
update_titles = sorted([title.replace("Архів:", "") for title in changes.keys()])

#def update_batch(updater, titles):
#    updater.update_records(titles)
#    updater.start_translation()


2026-01-14 16:56:54,951 [INFO] PageUpdateManager.init(): detect_environment==local
2026-01-14 16:56:55,224 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-01-14 16:56:55,225 [INFO] Runtime: truncating log history before 2025-11-15 23:56:55.225671+00:00


In [64]:
update_titles[3]

'Єврейське_містечко/Київська_губернія'

In [61]:
updater.update_records(update_titles[3])

2026-01-14 17:01:27,226 [INFO] Updater: accessing wiki page info for 1 pages
2026-01-14 17:01:28,121 [INFO] fetch_url: 8 requests in last 60s → 0.13 req/s
2026-01-14 17:01:28,665 [INFO] Updater: analyzing page links
2026-01-14 17:01:28,690 [INFO] Updater: linking child pages
2026-01-14 17:01:28,690 [INFO] Updater: accessing linked document metadata
2026-01-14 17:01:29,501 [INFO] skipping missing title: File:ДАКО_1459-1-35._1861_рік._Метрична_книга_євреїв_містечка_Новофастів_Сквирського_повіту._Шлюб.pdf, File:ДАКО 1459-1-35. 1861 рік. Метрична книга євреїв містечка Новофастів Сквирського повіту. Шлюб.pdf, {'ns': 6, 'title': 'File:ДАКО 1459-1-35. 1861 рік. Метрична книга євреїв містечка Новофастів Сквирського повіту. Шлюб.pdf', 'missing': '', 'imagerepository': ''}
2026-01-14 17:01:29,502 [INFO] _fetch_mediawiki_file_metadata_chunk: .... empty metadata: File:ДАКО_1459-1-35._1861_рік._Метрична_книга_євреїв_містечка_Новофастів_Сквирського_повіту._Шлюб.pdf, None
2026-01-14 17:01:38,144 [INF

True

In [62]:
x = _get_links(update_titles[3])
p = x["parse"]
p.keys()

2026-01-14 17:03:15,819 [INFO] fetch_url: 59 requests in last 60s → 0.98 req/s


dict_keys(['title', 'pageid', 'revid', 'links', 'images', 'externallinks', 'iwlinks', 'wikitext'])

In [63]:
len(p["iwlinks"])

2415

In [51]:
updater._get_page_info(update_titles[:10])

2026-01-14 16:47:53,636 [INFO] Updater: accessing wiki page info for 10 pages
2026-01-14 16:47:53,936 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s


[{'title': 'Єврейське_містечко',
  'record': {'description_uk': '',
   'years': '',
   'title': 'Єврейське_містечко',
   'level': 'fond',
   'label': None,
   'reference_date': '2025-11-20 12:06:43+00:00',
   'availability': 'linked',
   'source_type': 'wiki'},
  'links': {'title': 'Архів:Єврейське_містечко',
   'pageid': 130876,
   'parent': None,
   'children': [{'title': 'Архів:Єврейське_містечко/Волинська_губернія',
     'exists': True},
    {'title': 'Архів:Єврейське_містечко/Герцогство_Буковина', 'exists': True},
    {'title': 'Архів:Єврейське_містечко/Закарпаття', 'exists': True},
    {'title': 'Архів:Єврейське_містечко/Катеринославська_губернія',
     'exists': True},
    {'title': 'Архів:Єврейське_містечко/Київська_губернія', 'exists': True},
    {'title': 'Архів:Єврейське_містечко/Королівство_Галичини_та_Володимирії',
     'exists': True},
    {'title': 'Архів:Єврейське_містечко/Подільська_губернія', 'exists': True},
    {'title': 'Архів:Єврейське_містечко/Полтавська_губернія